# Full Simulator Render Farm: Round Robin, Q-Learning, DQN, dan PPO

Notebook Google Colab lengkap untuk penelitian **penjadwalan render farm animasi**. Desainnya bersifat online, non-preemptive, event-driven, dan menggunakan node heterogen.

**Keluaran utama**: raw results, tabel agregat, confidence interval, improvement terhadap Round Robin, Friedman test, Wilcoxon-Holm, effect size, ranking algoritma, grafik 300 dpi, model, manifest reproduksibilitas, dan ablation study opsional.

### Cara pakai
1. Jalankan profil `quick` untuk validasi.
2. Setelah semua quality check lulus, ubah ke profil `paper`.
3. Aktifkan Google Drive agar hasil tidak hilang saat runtime Colab terputus.
4. PPO dan ablation bersifat opsional karena lebih lama.

In [ ]:
#@title 1. Instalasi dependensi
!pip -q install "gymnasium==1.3.0" "stable-baselines3==2.9.0" "scipy>=1.11" "statsmodels>=0.14" "tqdm>=4.66" "rich>=13.7"

In [ ]:
#@title 2. Import dan konfigurasi
from __future__ import annotations
import os, sys, json, math, time, random, shutil, hashlib, platform, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import deque, defaultdict
from typing import Any, Dict, List, Optional, Sequence, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import BaseCallback
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
EXPERIMENT_PROFILE = "quick"  #@param ["quick", "paper"]
USE_GOOGLE_DRIVE = False       #@param {type:"boolean"}
RUN_DEEP_RL = True             #@param {type:"boolean"}
RUN_PPO = False                #@param {type:"boolean"}
RUN_ABLATION = False           #@param {type:"boolean"}
GLOBAL_SEED = 2026             #@param {type:"integer"}
RL_DEVICE = "cpu"             # CPU biasanya lebih cepat untuk MLP kecil; ubah ke "cuda" bila perlu
QUICK = EXPERIMENT_PROFILE == "quick"
assert EXPERIMENT_PROFILE in {"quick", "paper"}
def seed_all(seed:int):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    except Exception: pass
seed_all(GLOBAL_SEED)
try:
    import torch
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception: pass
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/render_farm_rl_q3')
else:
    ROOT=Path('/content/render_farm_rl_q3')
DIR={k:ROOT/k for k in ['data','models','results','tables','figures','logs']}
ROOT.mkdir(parents=True,exist_ok=True)
for p in DIR.values(): p.mkdir(parents=True,exist_ok=True)
import stable_baselines3
print('Profile:',EXPERIMENT_PROFILE,'| Root:',ROOT)
print('Python:',sys.version.split()[0],'Gymnasium:',gym.__version__,'SB3:',stable_baselines3.__version__)

## 3. Model simulator

Processing time job $i$ pada node $j$:

\[
T_{ij}=rac{B_i}{S_j}\epsilon_{ij}+rac{8D_i}{BW_j}
\]

$B_i$ adalah kebutuhan komputasi dasar, $S_j$ speed factor, $D_i$ ukuran data, $BW_j$ bandwidth, dan $\epsilon_{ij}$ noise log-normal.

In [ ]:
#@title 3. Struktur data dan generator workload
@dataclass(frozen=True)
class Scenario:
    n_nodes:int=4; n_jobs:int=100
    arrival_pattern:str='poisson'; heterogeneity:str='moderate'; job_mix:str='mixed'
    load_factor:float=.8; noise_sigma:float=.08; deadline_factor:float=5.; seed:int=42
    def validate(self):
        assert self.n_nodes>=2 and self.n_jobs>=2
        assert self.arrival_pattern in {'batch','poisson','burst'}
        assert self.heterogeneity in {'homogeneous','moderate','high'}
        assert self.job_mix in {'light','mixed','heavy'}
        assert self.load_factor>0
    @property
    def scenario_id(self):
        return hashlib.md5(json.dumps(asdict(self),sort_keys=True).encode()).hexdigest()[:12]
@dataclass
class Workload:
    cfg:Scenario; jobs:pd.DataFrame; nodes:pd.DataFrame
    processing:np.ndarray; compute:np.ndarray; transfer:np.ndarray
    def validate(self):
        assert self.processing.shape==(self.cfg.n_jobs,self.cfg.n_nodes)
        assert np.all(np.isfinite(self.processing)) and np.all(self.processing>0)
        assert np.all(np.diff(self.jobs.arrival_time)>=0)
RES={'480p':.45,'720p':1.,'1080p':2.25}
SAMP={64:.62,128:1.,256:1.62}
def complexities(rng,n,mix):
    if mix=='light': return rng.beta(2,5,n)
    if mix=='heavy': return rng.beta(5,2,n)
    g=rng.choice(3,n,p=[.3,.45,.25]); out=np.empty(n)
    for i,x in enumerate(g): out[i]=rng.beta(*((2,6) if x==0 else (3,3) if x==1 else (6,2)))
    return out
def make_nodes(cfg,rng):
    n=cfg.n_nodes
    if cfg.heterogeneity=='homogeneous': speed=rng.uniform(.95,1.05,n); bw=rng.uniform(900,1100,n)
    elif cfg.heterogeneity=='moderate':
        speed=rng.lognormal(0,.25,n); speed/=speed.mean(); bw=rng.lognormal(np.log(900),.25,n)
    else:
        speed=rng.lognormal(0,.5,n); speed/=speed.mean(); bw=rng.lognormal(np.log(800),.45,n)
    speed=np.clip(speed,.35,2.2); bw=np.clip(bw,150,2500)
    return pd.DataFrame({'node_id':range(n),'speed_factor':speed,'bandwidth_mbps':bw,
        'memory_gb':np.clip(np.round(8+8*speed+rng.normal(0,2,n)),8,32),
        'power_watts':np.clip(100+85*speed+rng.normal(0,8,n),100,320)})
def make_jobs(cfg,nodes,rng):
    n=cfg.n_jobs; comp=complexities(rng,n,cfg.job_mix)
    if cfg.job_mix=='light': rp,sp=[.55,.35,.1],[.55,.35,.1]
    elif cfg.job_mix=='heavy': rp,sp=[.1,.35,.55],[.1,.35,.55]
    else: rp,sp=[.25,.45,.3],[.25,.5,.25]
    r=rng.choice(list(RES),n,p=rp); s=rng.choice(list(SAMP),n,p=sp)
    rf=np.array([RES[x] for x in r]); sf=np.array([SAMP[int(x)] for x in s])
    base=18*rf*(.5+1.85*comp)*sf*rng.lognormal(0,.2,n); base=np.clip(base,2,420)
    size=(25+85*rf+80*comp+35*sf)*rng.lognormal(0,.12,n); size=np.clip(size,20,650)
    mem=np.clip(1+1.3*rf+2.4*comp+.9*sf,1.5,7.5)
    mean_service=float(np.mean(base/np.mean(nodes.speed_factor)))
    rate=max(1e-6,cfg.load_factor*float(nodes.speed_factor.sum())/mean_service); mean_ia=1/rate
    if cfg.arrival_pattern=='batch': arr=np.zeros(n)
    elif cfg.arrival_pattern=='poisson':
        ia=rng.exponential(mean_ia,n); ia[0]=0; arr=np.cumsum(ia)
    else:
        ia=np.empty(n); bs=max(5,n//10)
        for i in range(n): ia[i]=rng.exponential(mean_ia*(.22 if (i//bs)%2==0 else 2.2))
        ia[0]=0; arr=np.cumsum(ia)
    deadline=arr+cfg.deadline_factor*base*rng.uniform(.75,1.3,n)
    return pd.DataFrame({'job_id':range(n),'arrival_time':arr,'resolution':r,'resolution_factor':rf,
        'samples':s.astype(int),'sample_factor':sf,'complexity':comp,'base_compute_time':base,
        'input_size_mb':size,'memory_required_gb':mem,'deadline':deadline})
def generate_workload(cfg:Scenario):
    cfg.validate(); rng=np.random.default_rng(cfg.seed); nodes=make_nodes(cfg,rng); jobs=make_jobs(cfg,nodes,rng)
    speed=nodes.speed_factor.to_numpy()[None,:]; bw=nodes.bandwidth_mbps.to_numpy()[None,:]
    base=jobs.base_compute_time.to_numpy()[:,None]; size=jobs.input_size_mb.to_numpy()[:,None]
    noise=rng.lognormal(0,cfg.noise_sigma,(cfg.n_jobs,cfg.n_nodes))
    compute=base/speed*noise; transfer=size*8/bw; processing=compute+transfer
    w=Workload(cfg,jobs,nodes,processing,compute,transfer); w.validate(); return w
example=generate_workload(Scenario(seed=GLOBAL_SEED))
display(example.nodes.round(3)); display(example.jobs.head().round(3)); print(example.processing.shape)

## 4. Baseline scheduler dan discrete-event simulation

Baseline: Random, Round Robin, Least Loaded, Fastest Node, Earliest Finish, dan Balanced EFT.

In [ ]:
#@title 4. Baseline policies dan simulator
class Policy:
    name='Base'
    def reset(self,w,seed): self.workload=w; self.rng=np.random.default_rng(seed)
    def choose(self,i,arrival,remaining,queues,p_row,nodes): raise NotImplementedError
class RandomPolicy(Policy):
    name='Random'
    def choose(self,*a): return int(self.rng.integers(self.workload.cfg.n_nodes))
class RoundRobinPolicy(Policy):
    name='RoundRobin'
    def reset(self,w,seed): super().reset(w,seed); self.k=0
    def choose(self,*a): x=self.k%self.workload.cfg.n_nodes; self.k+=1; return x
class LeastLoadedPolicy(Policy):
    name='LeastLoaded'
    def choose(self,i,a,r,q,p,n): return int(np.argmin(r-1e-6*n.speed_factor.to_numpy()))
class FastestNodePolicy(Policy):
    name='FastestNode'
    def choose(self,i,a,r,q,p,n): return int(np.argmin(p))
class EarliestFinishPolicy(Policy):
    name='EarliestFinish'
    def choose(self,i,a,r,q,p,n): return int(np.argmin(r+p))
class BalancedEFTPolicy(Policy):
    name='BalancedEFT'
    def __init__(self,balance_weight=.3): self.balance_weight=balance_weight
    def choose(self,i,a,r,q,p,n):
        comp=r+p; scale=max(float(comp.min()),1e-9); score=[]
        for j in range(len(p)):
            x=r.copy(); x[j]+=p[j]; imb=float(np.std(x)/max(np.mean(x),1e-9)); score.append(comp[j]/scale+self.balance_weight*imb)
        return int(np.argmin(score))
BASELINES={'Random':RandomPolicy,'RoundRobin':RoundRobinPolicy,'LeastLoaded':LeastLoadedPolicy,
           'FastestNode':FastestNodePolicy,'EarliestFinish':EarliestFinishPolicy,'BalancedEFT':BalancedEFTPolicy}
def metrics_from_records(w,rec,busy,decision):
    makespan=max(float(rec.finish_time.max()-w.jobs.arrival_time.min()),1e-9); util_nodes=busy/makespan; power=w.nodes.power_watts.to_numpy()
    energy=float(np.sum(rec.processing_time.to_numpy()*power[rec.node_id.to_numpy(int)]/3600))
    return {'makespan':makespan,'avg_waiting_time':float(rec.waiting_time.mean()),'p95_waiting_time':float(rec.waiting_time.quantile(.95)),
      'avg_turnaround_time':float(rec.turnaround_time.mean()),'p95_turnaround_time':float(rec.turnaround_time.quantile(.95)),
      'throughput':float(len(rec)/makespan),'utilization':float(busy.sum()/(makespan*w.cfg.n_nodes)),
      'min_node_utilization':float(util_nodes.min()),'max_node_utilization':float(util_nodes.max()),
      'load_imbalance':float(np.std(busy)/max(np.mean(busy),1e-9)),
      'deadline_violation_rate':float(np.mean(rec.finish_time.to_numpy()>w.jobs.deadline.to_numpy())),
      'energy_wh':energy,'decision_overhead_ms':float(np.mean(decision)*1000 if decision else 0)}
def simulate(w,policy,seed=None,return_records=False):
    policy.reset(w,w.cfg.seed if seed is None else seed); n=w.cfg.n_nodes
    available=np.zeros(n); busy=np.zeros(n); fq=[deque() for _ in range(n)]; rows=[]; decision=[]
    for i,job in w.jobs.iterrows():
        a=float(job.arrival_time)
        for q in fq:
            while q and q[0]<=a+1e-12: q.popleft()
        rem=np.maximum(available-a,0); qs=np.array([len(q) for q in fq]); p=w.processing[i]
        t=time.perf_counter(); node=int(policy.choose(i,a,rem,qs,p,w.nodes)); decision.append(time.perf_counter()-t); assert 0<=node<n
        start=max(a,available[node]); pt=float(p[node]); finish=start+pt
        available[node]=finish; busy[node]+=pt; fq[node].append(finish)
        rows.append({'job_id':int(job.job_id),'node_id':node,'arrival_time':a,'start_time':start,'finish_time':finish,
                     'processing_time':pt,'waiting_time':start-a,'turnaround_time':finish-a})
    rec=pd.DataFrame(rows).sort_values('job_id',ignore_index=True); m=metrics_from_records(w,rec,busy,decision); m['algorithm']=policy.name
    return m,(rec if return_records else None)
for name,cls in BASELINES.items():
    m,_=simulate(example,cls()); print(f"{name:15s} makespan={m['makespan']:.2f} util={m['utilization']:.3f}")

## 5. Gymnasium environment

State berisi fitur job, remaining load, speed, bandwidth, queue length, progress, dan arrival time. Action adalah indeks node. Reward menggabungkan completion, waiting, imbalance, dan energy proxy.

In [ ]:
#@title 5. Custom Gymnasium environment
REWARD={'completion':.5,'waiting':.2,'imbalance':.2,'energy':.1}
class RenderFarmEnv(gym.Env):
    metadata={'render_modes':[]}
    def __init__(self,n_nodes=4,n_jobs=80,seed=42,fixed=None,patterns=('batch','poisson','burst'),hetero=('homogeneous','moderate','high'),mix=('light','mixed','heavy'),loads=(.6,.9,1.2),reward_weights=None,reward_mode='full',include_speed=True):
        super().__init__(); self.n_nodes=n_nodes; self.n_jobs=n_jobs; self.base_seed=seed; self.fixed=fixed
        self.patterns=tuple(patterns); self.hetero=tuple(hetero); self.mix=tuple(mix); self.loads=tuple(loads)
        self.rw=dict(REWARD if reward_weights is None else reward_weights); self.reward_mode=reward_mode; self.include_speed=include_speed
        if fixed is not None: assert fixed.cfg.n_nodes==n_nodes; self.n_jobs=fixed.cfg.n_jobs
        self.obs_dim=5+4*n_nodes+2; self.observation_space=spaces.Box(0.,1.,shape=(self.obs_dim,),dtype=np.float32); self.action_space=spaces.Discrete(n_nodes); self.rng=np.random.default_rng(seed)
    def sample_workload(self):
        if self.fixed is not None: return self.fixed
        return generate_workload(Scenario(self.n_nodes,self.n_jobs,str(self.rng.choice(self.patterns)),str(self.rng.choice(self.hetero)),str(self.rng.choice(self.mix)),float(self.rng.choice(self.loads)),seed=int(self.rng.integers(2_000_000_000))))
    def remaining(self,a):
        for q in self.fq:
            while q and q[0]<=a+1e-12: q.popleft()
        return np.maximum(self.available-a,0),np.array([len(q) for q in self.fq])
    def obs(self):
        if self.i>=self.n_jobs: return np.zeros(self.obs_dim,np.float32)
        j=self.w.jobs.iloc[self.i]; a=float(j.arrival_time); rem,qs=self.remaining(a); n=self.w.nodes
        jf=np.array([j.base_compute_time/self.time_scale,j.resolution_factor/max(RES.values()),j.complexity,j.sample_factor/max(SAMP.values()),j.input_size_mb/650])
        speed=n.speed_factor.to_numpy()/2.2; bw=n.bandwidth_mbps.to_numpy()/2500
        if not self.include_speed: speed*=0; bw*=0
        nf=np.column_stack([rem/self.horizon,speed,bw,qs/max(self.n_jobs,1)]).reshape(-1); g=np.array([self.i/max(self.n_jobs-1,1),a/self.horizon])
        return np.clip(np.r_[jf,nf,g],0,1).astype(np.float32)
    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        if seed is not None: self.rng=np.random.default_rng(seed)
        self.w=self.sample_workload(); self.n_jobs=self.w.cfg.n_jobs; self.i=0; self.available=np.zeros(self.n_nodes); self.busy=np.zeros(self.n_nodes); self.fq=[deque() for _ in range(self.n_nodes)]; self.rows=[]
        self.time_scale=max(60.,float(self.w.jobs.base_compute_time.quantile(.95))); self.horizon=max(60.,float(self.w.jobs.base_compute_time.sum())/self.n_nodes)
        return self.obs(),{'scenario':asdict(self.w.cfg)}
    def step(self,action):
        action=int(action); assert self.action_space.contains(action); j=self.w.jobs.iloc[self.i]; a=float(j.arrival_time); rem,_=self.remaining(a); p=self.w.processing[self.i]
        start=max(a,self.available[action]); pt=float(p[action]); finish=start+pt; wait=start-a; turn=finish-a; proj=rem.copy(); proj[action]+=pt
        imb=float(np.std(proj)/max(np.mean(proj),1e-9)); energy=pt*float(self.w.nodes.iloc[action].power_watts)/3600
        if self.reward_mode=='makespan_only': reward=-(turn/self.time_scale)
        else: reward=-(self.rw['completion']*turn/self.time_scale+self.rw['waiting']*wait/self.time_scale+self.rw['imbalance']*imb+self.rw['energy']*energy/max(1,self.time_scale*250/3600))
        self.available[action]=finish; self.busy[action]+=pt; self.fq[action].append(finish)
        self.rows.append({'job_id':int(j.job_id),'node_id':action,'arrival_time':a,'start_time':start,'finish_time':finish,'processing_time':pt,'waiting_time':wait,'turnaround_time':turn})
        self.i+=1; terminated=self.i>=self.n_jobs; info={}
        if terminated:
            rec=pd.DataFrame(self.rows).sort_values('job_id',ignore_index=True); m=metrics_from_records(self.w,rec,self.busy,[]); reward+=-.2*m['makespan']/self.horizon+.1*m['utilization']; info={'episode_metrics':m,'records':rec}
        return self.obs(),float(reward),terminated,False,info
    def discrete_state(self):
        if self.i>=self.n_jobs: return (0,0,0,4)
        j=self.w.jobs.iloc[self.i]; rem,_=self.remaining(float(j.arrival_time)); jb=int(np.digitize(float(j.base_compute_time)/self.time_scale,[.55,1.2])); least=int(np.argmin(rem)); imb=float(np.std(rem)/max(np.mean(rem),1e-9)) if np.mean(rem)>0 else 0
        return (jb,least,int(np.digitize(imb,[.2,.6])),min(4,int(5*self.i/max(self.n_jobs,1))))
check_env(RenderFarmEnv(n_nodes=4,n_jobs=30,seed=GLOBAL_SEED),warn=True); print('Environment check: OK')

In [ ]:
#@title 6. Tabular Q-Learning
class QAgent:
    def __init__(self,n_actions,alpha=.12,gamma=.98,seed=42): self.n_actions=n_actions; self.alpha=alpha; self.gamma=gamma; self.rng=np.random.default_rng(seed); self.q=defaultdict(lambda:np.zeros(n_actions))
    def act(self,s,eps=0):
        if self.rng.random()<eps: return int(self.rng.integers(self.n_actions))
        v=self.q[s]; return int(self.rng.choice(np.flatnonzero(np.isclose(v,v.max()))))
    def update(self,s,a,r,ns,done):
        target=r if done else r+self.gamma*self.q[ns].max(); self.q[s][a]+=self.alpha*(target-self.q[s][a])
def train_q(n_nodes,n_jobs,episodes,seed):
    env=RenderFarmEnv(n_nodes,n_jobs,seed); ag=QAgent(n_nodes,seed=seed); hist=[]
    for ep in tqdm(range(episodes),desc=f'Q {n_nodes} nodes'):
        eps=.05+.95*math.exp(-5*ep/max(episodes,1)); env.reset(seed=seed+ep); s=env.discrete_state(); done=False; ret=0; info={}
        while not done:
            a=ag.act(s,eps); _,r,t,tr,info=env.step(a); done=t or tr; ns=env.discrete_state(); ag.update(s,a,r,ns,done); s=ns; ret+=r
        hist.append({'episode':ep+1,'return':ret,'epsilon':eps,**info['episode_metrics']})
    return ag,pd.DataFrame(hist)
def save_q(ag,path):
    obj={'n_actions':ag.n_actions,'alpha':ag.alpha,'gamma':ag.gamma,'q_table':{'|'.join(map(str,k)):v.tolist() for k,v in ag.q.items()}}; Path(path).write_text(json.dumps(obj),encoding='utf-8')
def eval_q(ag,w):
    env=RenderFarmEnv(w.cfg.n_nodes,w.cfg.n_jobs,w.cfg.seed,fixed=w); env.reset(seed=w.cfg.seed); done=False; info={}; dt=[]
    while not done:
        s=env.discrete_state(); t=time.perf_counter(); a=ag.act(s,0); dt.append(time.perf_counter()-t); _,_,te,tr,info=env.step(a); done=te or tr
    m=dict(info['episode_metrics']); m['algorithm']='QLearning'; m['decision_overhead_ms']=np.mean(dt)*1000; return m

In [ ]:
#@title 7. DQN/PPO helpers
class MetricsCallback(BaseCallback):
    def __init__(self): super().__init__(0); self.rows=[]
    def _on_step(self):
        for info in self.locals.get('infos',[]):
            if 'episode_metrics' in info: self.rows.append({'timesteps':self.num_timesteps,**info['episode_metrics']})
        return True
def train_dqn(n_nodes,n_jobs,steps,seed,path,reward_mode='full',include_speed=True,reward_weights=None):
    env=RenderFarmEnv(n_nodes,n_jobs,seed,reward_mode=reward_mode,include_speed=include_speed,reward_weights=reward_weights); cb=MetricsCallback()
    model=DQN('MlpPolicy',env,learning_rate=8e-4,buffer_size=50_000 if QUICK else 150_000,learning_starts=500 if QUICK else 3000,batch_size=128,gamma=.99,train_freq=4,gradient_steps=1,target_update_interval=1000,exploration_fraction=.35,exploration_final_eps=.04,policy_kwargs=dict(net_arch=[128,128]),seed=seed,device=RL_DEVICE,verbose=0)
    model.learn(steps,callback=cb,progress_bar=True); model.save(str(path)); return model,pd.DataFrame(cb.rows)
def train_ppo(n_nodes,n_jobs,steps,seed,path):
    env=RenderFarmEnv(n_nodes,n_jobs,seed); cb=MetricsCallback(); model=PPO('MlpPolicy',env,learning_rate=3e-4,n_steps=256 if QUICK else 512,batch_size=64 if QUICK else 128,n_epochs=10,gamma=.99,gae_lambda=.95,clip_range=.2,ent_coef=.01,policy_kwargs=dict(net_arch=[128,128]),seed=seed,device=RL_DEVICE,verbose=0)
    model.learn(steps,callback=cb,progress_bar=True); model.save(str(path)); return model,pd.DataFrame(cb.rows)
def eval_sb3(model,w,name,env_kwargs=None):
    env_kwargs=env_kwargs or {}; env=RenderFarmEnv(w.cfg.n_nodes,w.cfg.n_jobs,w.cfg.seed,fixed=w,**env_kwargs); obs,_=env.reset(seed=w.cfg.seed); done=False; info={}; dt=[]
    while not done:
        t=time.perf_counter(); a,_=model.predict(obs,deterministic=True); dt.append(time.perf_counter()-t); obs,_,te,tr,info=env.step(int(a)); done=te or tr
    m=dict(info['episode_metrics']); m['algorithm']=name; m['decision_overhead_ms']=np.mean(dt)*1000; return m

In [ ]:
#@title 8. Training model
if QUICK: NODE_COUNTS=[4]; TRAIN_JOBS=60; Q_EPISODES=250; DQN_STEPS=15_000; PPO_STEPS=20_000
else: NODE_COUNTS=[4,8,16]; TRAIN_JOBS=180; Q_EPISODES=4000; DQN_STEPS=180_000; PPO_STEPS=240_000
q_agents={}; dqn_models={}; ppo_models={}; histories=[]
for n in NODE_COUNTS:
    q,h=train_q(n,TRAIN_JOBS,Q_EPISODES,GLOBAL_SEED+n); q_agents[n]=q; save_q(q,DIR['models']/f'q_learning_{n}_nodes.json'); h['algorithm']='QLearning'; h['n_nodes']=n; histories.append(h)
    if RUN_DEEP_RL:
        m,h=train_dqn(n,TRAIN_JOBS,DQN_STEPS,GLOBAL_SEED+100+n,DIR['models']/f'dqn_{n}_nodes'); dqn_models[n]=m; h['algorithm']='DQN'; h['n_nodes']=n; histories.append(h)
        if RUN_PPO:
            m,h=train_ppo(n,TRAIN_JOBS,PPO_STEPS,GLOBAL_SEED+200+n,DIR['models']/f'ppo_{n}_nodes'); ppo_models[n]=m; h['algorithm']='PPO'; h['n_nodes']=n; histories.append(h)
train_hist=pd.concat(histories,ignore_index=True,sort=False); train_hist.to_csv(DIR['results']/f'training_history_{EXPERIMENT_PROFILE}.csv',index=False); print(train_hist.shape)

In [ ]:
#@title 9. Kurva konvergensi
fig,ax=plt.subplots(figsize=(9,5))
for (alg,n),g in train_hist.groupby(['algorithm','n_nodes']):
    x=g['episode'] if alg=='QLearning' else g['timesteps']; y=g['makespan'].rolling(max(3,len(g)//25),min_periods=1).mean(); ax.plot(x,y,label=f'{alg}-{n}')
ax.set(title='Training convergence',xlabel='Episode / timesteps',ylabel='Makespan'); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(DIR['figures']/f'training_convergence_{EXPERIMENT_PROFILE}.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
#@title 10. Grid evaluasi
if QUICK: NS=[4]; JS=[100]; PAT=['poisson','burst']; HET=['homogeneous','moderate']; MIX=['mixed']; LOAD=[.7,1.]; SEEDS=[101,102,103]
else: NS=[4,8,16]; JS=[250,1000]; PAT=['batch','poisson','burst']; HET=['homogeneous','moderate','high']; MIX=['mixed']; LOAD=[.6,.9,1.2]; SEEDS=list(range(1001,1031))
grid=[]
for n in NS:
 for j in JS:
  for p in PAT:
   loads=[1.] if p=='batch' else LOAD
   for h in HET:
    for m in MIX:
     for l in loads:
      for s in SEEDS: grid.append(Scenario(n,j,p,h,m,l,seed=s))
print('Unique workloads:',len(grid))

In [ ]:
#@title 11. Batch experiment
rows=[]; raw_path=DIR['results']/f'raw_results_{EXPERIMENT_PROFILE}.csv'
for k,cfg in enumerate(tqdm(grid,desc='Evaluation'),1):
    w=generate_workload(cfg); meta={'scenario_id':cfg.scenario_id,**asdict(cfg)}
    for _,cls in BASELINES.items(): m,_=simulate(w,cls(),cfg.seed); rows.append({**meta,**m})
    if cfg.n_nodes in q_agents: rows.append({**meta,**eval_q(q_agents[cfg.n_nodes],w)})
    if RUN_DEEP_RL and cfg.n_nodes in dqn_models: rows.append({**meta,**eval_sb3(dqn_models[cfg.n_nodes],w,'DQN')})
    if RUN_DEEP_RL and RUN_PPO and cfg.n_nodes in ppo_models: rows.append({**meta,**eval_sb3(ppo_models[cfg.n_nodes],w,'PPO')})
    if k%(10 if QUICK else 50)==0: pd.DataFrame(rows).to_csv(raw_path,index=False)
results=pd.DataFrame(rows); results.to_csv(raw_path,index=False); display(results.head()); print(results.shape)

In [ ]:
#@title 12. Quality checks
metrics=['makespan','avg_waiting_time','p95_waiting_time','avg_turnaround_time','throughput','utilization','load_imbalance','deadline_violation_rate','energy_wh']
assert not results.empty and results[metrics].notna().all().all(); assert (results.makespan>0).all() and (results.avg_waiting_time>=-1e-9).all(); assert results.utilization.between(0,1+1e-9).all() and results.deadline_violation_rate.between(0,1).all(); assert results.groupby('scenario_id').algorithm.nunique().nunique()==1
print('QUALITY CHECKS: LULUS | Algorithms:',sorted(results.algorithm.unique()))

In [ ]:
#@title 13. Descriptive statistics dan 95% CI
METRICS=['makespan','avg_waiting_time','p95_waiting_time','avg_turnaround_time','p95_turnaround_time','throughput','utilization','load_imbalance','deadline_violation_rate','energy_wh','decision_overhead_ms']
def aggregate_ci(df,group_cols,metrics):
    out=[]
    for keys,g in df.groupby(group_cols,dropna=False):
        keys=keys if isinstance(keys,tuple) else (keys,); base=dict(zip(group_cols,keys))
        for metric in metrics:
            x=g[metric].dropna().to_numpy(float); mean=x.mean(); sd=x.std(ddof=1) if len(x)>1 else 0; sem=sd/math.sqrt(max(len(x),1)); tc=stats.t.ppf(.975,max(len(x)-1,1)) if len(x)>1 else 0
            out.append({**base,'metric':metric,'n':len(x),'mean':mean,'std':sd,'ci95_low':mean-tc*sem,'ci95_high':mean+tc*sem})
    return pd.DataFrame(out)
summary=aggregate_ci(results,['algorithm','n_nodes','n_jobs','arrival_pattern','heterogeneity','load_factor'],METRICS); summary.to_csv(DIR['tables']/f'descriptive_summary_{EXPERIMENT_PROFILE}.csv',index=False)
overall=results.groupby('algorithm')[METRICS].agg(['mean','std','median']); overall.to_csv(DIR['tables']/f'overall_summary_{EXPERIMENT_PROFILE}.csv'); display(overall.round(4))
rr=results.query("algorithm=='RoundRobin'")[['scenario_id','makespan']].rename(columns={'makespan':'rr_makespan'}); imp=results.merge(rr,on='scenario_id'); imp['improvement_pct']=100*(imp.rr_makespan-imp.makespan)/imp.rr_makespan
imp_summary=imp.groupby('algorithm').improvement_pct.agg(['mean','median','std','count']).sort_values('mean',ascending=False); imp_summary.to_csv(DIR['tables']/f'improvement_vs_rr_{EXPERIMENT_PROFILE}.csv'); display(imp_summary.round(3))

In [ ]:
#@title 14. Friedman, Wilcoxon-Holm, effect size, ranking
def rank_biserial(ref,cand):
    d=ref-cand; d=d[np.abs(d)>1e-12]
    if len(d)==0:return 0.
    r=stats.rankdata(np.abs(d)); pos=r[d>0].sum(); neg=r[d<0].sum(); return float((pos-neg)/max(pos+neg,1e-9))
piv=results.pivot_table(index='scenario_id',columns='algorithm',values='makespan',aggfunc='first').dropna(); algs=list(piv.columns); fs,fp=stats.friedmanchisquare(*[piv[a].to_numpy() for a in algs])
friedman=pd.DataFrame([{'metric':'makespan','n_blocks':len(piv),'n_algorithms':len(algs),'chi_square':fs,'p_value':fp}]); friedman.to_csv(DIR['tables']/f'friedman_{EXPERIMENT_PROFILE}.csv',index=False)
ref=piv.RoundRobin.to_numpy(); wr=[]
for a in algs:
    if a=='RoundRobin':continue
    x=piv[a].to_numpy()
    try: st,p=stats.wilcoxon(ref,x,zero_method='wilcox')
    except ValueError: st,p=0,1
    wr.append({'algorithm':a,'statistic':st,'p_raw':p,'median_improvement_pct':np.median(100*(ref-x)/ref),'rank_biserial':rank_biserial(ref,x)})
wil=pd.DataFrame(wr); reject,padj,_,_=multipletests(wil.p_raw,alpha=.05,method='holm'); wil['p_holm']=padj; wil['significant']=reject; wil.to_csv(DIR['tables']/f'wilcoxon_holm_{EXPERIMENT_PROFILE}.csv',index=False)
ranks=piv.rank(axis=1); ranking=pd.DataFrame({'algorithm':ranks.columns,'mean_rank':ranks.mean().to_numpy(),'median_rank':ranks.median().to_numpy()}).sort_values('mean_rank'); ranking.to_csv(DIR['tables']/f'algorithm_ranking_{EXPERIMENT_PROFILE}.csv',index=False)
display(friedman); display(wil.sort_values('p_holm').round(5)); display(ranking.round(3))

In [ ]:
#@title 15. Publication figures
plot=imp.copy(); plot['relative_pct']=100*plot.makespan/plot.rr_makespan; order=plot.groupby('algorithm').relative_pct.median().sort_values().index
fig,ax=plt.subplots(figsize=(11,5.5)); ax.boxplot([plot.loc[plot.algorithm==a,'relative_pct'] for a in order],labels=order,showfliers=False); ax.axhline(100,ls='--'); ax.tick_params(axis='x',rotation=35); ax.set(title='Relative makespan across identical workloads',ylabel='Relative to Round Robin (%)'); ax.grid(axis='y',alpha=.25); fig.tight_layout(); fig.savefig(DIR['figures']/f'relative_makespan_{EXPERIMENT_PROFILE}.png',dpi=300,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(8,5)); rp=ranking.sort_values('mean_rank'); ax.barh(rp.algorithm,rp.mean_rank); ax.invert_yaxis(); ax.set(title='Mean rank (lower is better)',xlabel='Mean rank'); ax.grid(axis='x',alpha=.25); fig.tight_layout(); fig.savefig(DIR['figures']/f'mean_rank_{EXPERIMENT_PROFILE}.png',dpi=300,bbox_inches='tight'); plt.show()
sc=results.groupby(['algorithm','n_nodes'],as_index=False).makespan.mean(); fig,ax=plt.subplots(figsize=(9,5))
for a,g in sc.groupby('algorithm'): ax.plot(g.n_nodes,g.makespan,marker='o',label=a)
ax.set(title='Scalability by number of nodes',xlabel='Nodes',ylabel='Mean makespan'); ax.grid(alpha=.25); ax.legend(ncol=2); fig.tight_layout(); fig.savefig(DIR['figures']/f'scalability_{EXPERIMENT_PROFILE}.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
#@title 16. Robustness table dan article-ready table
robust=results.groupby(['algorithm','arrival_pattern','heterogeneity','load_factor'],as_index=False).agg(makespan_mean=('makespan','mean'),waiting_mean=('avg_waiting_time','mean'),utilization_mean=('utilization','mean'),deadline_violation_mean=('deadline_violation_rate','mean'),n=('scenario_id','count')); robust.to_csv(DIR['tables']/f'robustness_{EXPERIMENT_PROFILE}.csv',index=False)
article=results.groupby('algorithm').agg(makespan_mean=('makespan','mean'),makespan_std=('makespan','std'),waiting_mean=('avg_waiting_time','mean'),waiting_std=('avg_waiting_time','std'),utilization_mean=('utilization','mean'),utilization_std=('utilization','std'),imbalance_mean=('load_imbalance','mean'),deadline_violation_mean=('deadline_violation_rate','mean'),energy_mean=('energy_wh','mean')).reset_index()
article=article.merge(imp_summary[['mean','median']].rename(columns={'mean':'improvement_mean_pct','median':'improvement_median_pct'}),left_on='algorithm',right_index=True).sort_values('makespan_mean'); article.to_csv(DIR['tables']/f'article_main_results_{EXPERIMENT_PROFILE}.csv',index=False); (DIR['tables']/f'article_main_results_{EXPERIMENT_PROFILE}.tex').write_text(article.round(4).to_latex(index=False,escape=True),encoding='utf-8'); display(article.round(4))

In [ ]:
#@title 17. Optional DQN ablation study
ablation=pd.DataFrame()
if RUN_ABLATION and RUN_DEEP_RL:
    n=4 if QUICK else 8; jobs=60 if QUICK else 180; steps=12_000 if QUICK else 120_000; seeds=[701,702,703] if QUICK else list(range(701,731))
    variants={'full':dict(reward_mode='full',include_speed=True,reward_weights=REWARD),'makespan_only':dict(reward_mode='makespan_only',include_speed=True,reward_weights=REWARD),'no_speed':dict(reward_mode='full',include_speed=False,reward_weights=REWARD),'no_imbalance':dict(reward_mode='full',include_speed=True,reward_weights={'completion':.6,'waiting':.25,'imbalance':0,'energy':.15})}
    models={}
    for name,kw in variants.items(): models[name],_=train_dqn(n,jobs,steps,GLOBAL_SEED+900+n,DIR['models']/f'ablation_{name}_{n}',**kw)
    ar=[]
    for s in tqdm(seeds,desc='Ablation eval'):
        w=generate_workload(Scenario(n,250 if QUICK else 1000,'burst','high','mixed',1.2,seed=s))
        for name,model in models.items(): ar.append({'variant':name,'seed':s,**eval_sb3(model,w,name,variants[name])})
    ablation=pd.DataFrame(ar); ablation.to_csv(DIR['tables']/f'ablation_{EXPERIMENT_PROFILE}.csv',index=False); display(ablation.groupby('variant')[['makespan','avg_waiting_time','utilization','load_imbalance']].agg(['mean','std']))
else: print('Ablation skipped. Set RUN_ABLATION=True.')

In [ ]:
#@title 18. Representative workload, manifest, dan ZIP hasil
rep=generate_workload(Scenario(NS[0],JS[0],'burst','moderate','mixed',1.,seed=999)); rep.jobs.to_csv(DIR['data']/f'representative_jobs_{EXPERIMENT_PROFILE}.csv',index=False); rep.nodes.to_csv(DIR['data']/f'representative_nodes_{EXPERIMENT_PROFILE}.csv',index=False); np.save(DIR['data']/f'representative_processing_{EXPERIMENT_PROFILE}.npy',rep.processing)
try:
 import torch; device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'; tv=torch.__version__
except Exception: device='unknown'; tv='unknown'
manifest={'created_at_utc':pd.Timestamp.utcnow().isoformat(),'profile':EXPERIMENT_PROFILE,'seed':GLOBAL_SEED,'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,'gymnasium':gym.__version__,'stable_baselines3':stable_baselines3.__version__,'torch':tv,'device':device,'grid':{'nodes':NS,'jobs':JS,'patterns':PAT,'heterogeneity':HET,'mix':MIX,'loads':LOAD,'seeds':SEEDS},'reward':REWARD}
(ROOT/f'manifest_{EXPERIMENT_PROFILE}.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8'); zip_path=shutil.make_archive(f'/content/render_farm_rl_q3_{EXPERIMENT_PROFILE}','zip',ROOT); print('Results:',ROOT); print('ZIP:',zip_path)

## 19. Interpretasi dan threats to validity

Klaim keunggulan harus didukung **nilai rata-rata, 95% CI, Wilcoxon-Holm, dan effect size**. Laporkan juga skenario ketika RL kalah dari heuristic.

**Threats to validity**:
- Processing time berasal dari model sintetis, bukan trace industri.
- Job diasumsikan independen, non-preemptive, dan FIFO per node.
- Node failure, dependency antarframe, cache, dan network congestion belum dimodelkan penuh.
- Generalisasi dibatasi rentang parameter simulator.

Untuk memperkuat artikel Q3, kalibrasikan `base_compute_time`, faktor resolusi, dan noise menggunakan 30–100 frame Blender pada satu komputer.

### Struktur bagian hasil artikel
1. Experimental Configuration
2. Training Convergence
3. Overall Scheduling Performance
4. Heterogeneous-node Analysis
5. Dynamic and Burst Workload Analysis
6. Statistical Significance and Effect Size
7. Ablation Study
8. Computational Overhead
9. Threats to Validity

## 20. Referensi implementasi
- Gymnasium custom environment: https://gymnasium.farama.org/introduction/create_custom_env/
- Gymnasium API: https://gymnasium.farama.org/api/env/
- Stable-Baselines3 custom environment: https://stable-baselines3.readthedocs.io/en/master/guide/custom_env.html
- Stable-Baselines3 algorithms: https://stable-baselines3.readthedocs.io/en/master/guide/algos.html
- Google Colab FAQ: https://research.google.com/colaboratory/faq.html